In [0]:
from  pyspark.sql import functions as F

In [0]:
# Captura sinais de demanda ao longo do tempo (série temporal)
# Útil para modelos de previsão de venda de ingressos

fact_tickets = spark.table("gold_bi.fact_ticket_sales")
dim_dates    = spark.table("gold_bi.dim_dates")

demand_ts = (
    fact_tickets
    .join(dim_dates, "date_key")
    .groupBy("event_id", "date_key", "full_date", "day_of_week", "is_weekend")
    .agg(
        F.count("order_id").alias("orders_on_day"),
        F.sum("unit_price_brl").alias("revenue_on_day"),
    )
    # Calcula dias restantes até o evento para cada ponto na série
    .join(spark.table("silver.events").select("event_id", "start_date"), "event_id")
    .withColumn("days_to_event",
        F.datediff(F.col("start_date"), F.col("full_date")))
    .filter(F.col("days_to_event") >= 0)
    .orderBy("event_id", "date_key")
)

(demand_ts.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_ai.features_demand_signals"))